# 02 · Optimisation and transfer learning

*Intermediate Model Training*, notebook 2 of 4 · lessons 3 and 4: **Control learning rates, clipping and accumulation** and **Adapt pretrained representations**

An optimiser turns gradients into weight updates, and three controls decide how: the learning rate at each update, a limit on how large one update may be, and how many examples contribute to it. You will build all three, prove that gradient accumulation gives exactly the large-batch gradient, and then use the trained network for something new: a second task with only 40 labelled examples, solved by reusing what the network learned on the first.

## What you will build

- `warmup_cosine_lr(step, total_steps, peak_lr, warmup_steps, min_lr=0.0)`: a learning-rate schedule driven by the optimiser-update clock;
- `clip_by_global_norm(grads, max_norm)`: one scale factor for all gradients when their combined norm is too large;
- `accumulated_gradients(params, loss_fn, X, y, micro_batch_size)`: a large batch's gradient computed one micro-batch at a time, short last micro-batch included;
- `extract_features(model, X)`: frozen features from a pretrained network, for a new head trained on very few labels.

## Learning objectives

By the end of this notebook you will be able to:

1. implement linear warm-up followed by cosine decay, and advance a schedule once per optimiser update;
2. clip gradients by their global norm, log that norm, and tell a clipped symptom from its cause;
3. accumulate gradients over micro-batches so that the update equals the large-batch update, also when the last micro-batch is short;
4. pretrain a network on a related task, freeze it, and train a new head on few labels, then compare it with training from scratch;
5. explain what freezing changes (no gradients, no updates) and what it does not (dropout and other train-mode behaviour).

### How to work through this notebook

Run the cells from top to bottom. When you reach a **✍️ Your turn** cell, type the function yourself and then run the check cell beneath it:

- ✅ means your function matches the reference, and from then on the notebook runs **your** code;
- ✍️ means you have not written it yet, and ❌ means it does not match yet. In both cases the notebook quietly uses a hidden reference version, so you can always keep going and come back later.

Each reference solution sits in a collapsed cell just above the function you type. Open it only when you are stuck.

## Dependencies

- **Tier 1:** NumPy for every calculation and Matplotlib for the plots. Nothing else is imported, so the notebook runs in JupyterLite in your browser, on the course JupyterHub server and in Google Colab.
- Nothing is downloaded: both tasks are generated in the notebook from fixed seeds.
- The networks use the tiny tensor engine and the `MLP` from Base Model Training, pasted into hidden recap cells. This notebook updates the weights with plain SGD with momentum instead of Adam, so the learning rate you set is exactly the step size you get.
- The whole notebook runs in a few seconds on a laptop; in the browser it takes a little longer.

## Setup

Run this cell first. It imports NumPy and Matplotlib, creates a seeded random generator and defines `practice`, the helper that checks each function you write against a reference.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import numpy as np

rng = np.random.default_rng(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    try:
        got_array, want_array = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    except (TypeError, ValueError):
        return got == want
    return got_array.shape == want_array.shape and np.allclose(got_array, want_array, atol=atol)


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    if isinstance(value, np.ndarray):
        text = np.array2string(value, precision=4, threshold=8, separator=", ").replace("\n", "")
    else:
        text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-6):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours

import time

import matplotlib.pyplot as plt

## 📦 Recap: the tiny tensor engine you built in Base Model Training

The hidden cell below is the reverse-mode autograd engine from Base Model Training, pasted unchanged: `Tensor` (with `+`, `-`, `*`, `/`, `**`, `@`, `exp`, `log`, `tanh`, `relu`, `sigmoid`, `sum`, `mean`, `reshape`, `transpose`, indexing, `log_softmax`, `softmax` and `backward`), `parameter`, `zeros_parameter`, `zero_grad`, `cross_entropy`, `layer_norm`, `numerical_grad` and `relative_error`. Every operation in it was checked against numerical gradients. Run the cell; open it only if you want to look back at the code.

In [ ]:
# ── Tiny tensor autograd (NumPy) ── the engine you built in Base Model Training ──
import numpy as np


def _unbroadcast(grad, shape):
    """Sum a gradient back down to `shape`, undoing NumPy broadcasting."""
    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)
    for axis, size in enumerate(shape):
        if size == 1 and grad.shape[axis] != 1:
            grad = grad.sum(axis=axis, keepdims=True)
    return grad


class Tensor:
    """A NumPy array that remembers how it was made, so gradients can flow back."""

    def __init__(self, data, requires_grad=False, parents=(), backward=None, op=""):
        self.data = np.asarray(data, dtype=np.float64)
        self.requires_grad = requires_grad
        self.grad = None
        self._parents = parents
        self._backward = backward  # grad_out -> one gradient per parent
        self.op = op

    shape = property(lambda self: self.data.shape)
    ndim = property(lambda self: self.data.ndim)
    T = property(lambda self: self.transpose())

    def __repr__(self):
        return f"Tensor(shape={self.shape}, op={self.op or 'leaf'}, requires_grad={self.requires_grad})"

    @staticmethod
    def lift(value):
        return value if isinstance(value, Tensor) else Tensor(value)

    def _result(self, data, parents, backward, op):
        if any(parent.requires_grad for parent in parents):
            return Tensor(data, True, parents, backward, op)
        return Tensor(data, op=op)

    # arithmetic, with broadcasting
    def __add__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data + other.data, (self, other), lambda g: (g, g), "add")

    __radd__ = __add__

    def __neg__(self):
        return self._result(-self.data, (self,), lambda g: (-g,), "neg")

    def __sub__(self, other):
        return self + (-Tensor.lift(other))

    def __rsub__(self, other):
        return Tensor.lift(other) + (-self)

    def __mul__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data * other.data, (self, other),
                            lambda g: (g * other.data, g * self.data), "mul")

    __rmul__ = __mul__

    def __truediv__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data / other.data, (self, other),
                            lambda g: (g / other.data, -g * self.data / other.data ** 2), "div")

    def __rtruediv__(self, other):
        return Tensor.lift(other) / self

    def __pow__(self, exponent):
        assert isinstance(exponent, (int, float)), "only constant exponents"
        return self._result(self.data ** exponent, (self,),
                            lambda g: (g * exponent * self.data ** (exponent - 1),), "pow")

    def __matmul__(self, other):
        other = Tensor.lift(other)

        def backward(g):
            return (g @ np.swapaxes(other.data, -1, -2), np.swapaxes(self.data, -1, -2) @ g)

        return self._result(self.data @ other.data, (self, other), backward, "matmul")

    # elementwise functions
    def exp(self):
        out = np.exp(self.data)
        return self._result(out, (self,), lambda g: (g * out,), "exp")

    def log(self):
        return self._result(np.log(self.data), (self,), lambda g: (g / self.data,), "log")

    def tanh(self):
        out = np.tanh(self.data)
        return self._result(out, (self,), lambda g: (g * (1 - out ** 2),), "tanh")

    def relu(self):
        return self._result(np.maximum(self.data, 0), (self,), lambda g: (g * (self.data > 0),), "relu")

    def sigmoid(self):
        out = 0.5 * (1 + np.tanh(0.5 * self.data))  # stable for large |x|
        return self._result(out, (self,), lambda g: (g * out * (1 - out),), "sigmoid")

    # reductions and reshaping
    def sum(self, axis=None, keepdims=False):
        def backward(g):
            if axis is not None and not keepdims:
                g = np.expand_dims(g, axis)
            return (np.broadcast_to(g, self.shape).copy(),)

        return self._result(self.data.sum(axis=axis, keepdims=keepdims), (self,), backward, "sum")

    def mean(self, axis=None, keepdims=False):
        count = self.data.size if axis is None else np.prod([self.shape[a] for a in np.atleast_1d(axis)])
        return self.sum(axis=axis, keepdims=keepdims) / count

    def reshape(self, *shape):
        return self._result(self.data.reshape(*shape), (self,), lambda g: (g.reshape(self.shape),), "reshape")

    def transpose(self, *axes):
        axes = axes or tuple(reversed(range(self.ndim)))
        inverse = np.argsort(axes)
        return self._result(self.data.transpose(axes), (self,), lambda g: (g.transpose(inverse),), "transpose")

    def __getitem__(self, index):
        def backward(g):
            full = np.zeros_like(self.data)
            np.add.at(full, index, g)  # repeated indices (an embedding row used twice) add up
            return (full,)

        return self._result(self.data[index], (self,), backward, "index")

    def log_softmax(self, axis=-1):
        shifted = self.data - self.data.max(axis=axis, keepdims=True)
        out = shifted - np.log(np.exp(shifted).sum(axis=axis, keepdims=True))
        probs = np.exp(out)
        return self._result(out, (self,), lambda g: (g - probs * g.sum(axis=axis, keepdims=True),), "log_softmax")

    def softmax(self, axis=-1):
        shifted = np.exp(self.data - self.data.max(axis=axis, keepdims=True))
        out = shifted / shifted.sum(axis=axis, keepdims=True)
        return self._result(out, (self,), lambda g: (out * (g - (g * out).sum(axis=axis, keepdims=True)),), "softmax")

    # reverse-mode differentiation
    def backward(self, grad=None):
        order, seen = [], set()

        def visit(node):
            if id(node) not in seen:
                seen.add(id(node))
                for parent in node._parents:
                    visit(parent)
                order.append(node)

        visit(self)
        self.grad = np.ones_like(self.data) if grad is None else np.asarray(grad, dtype=np.float64)
        for node in reversed(order):
            if node._backward is None or node.grad is None:
                continue
            for parent, parent_grad in zip(node._parents, node._backward(node.grad)):
                if not parent.requires_grad or parent_grad is None:
                    continue
                parent_grad = _unbroadcast(np.asarray(parent_grad, dtype=np.float64), parent.shape)
                parent.grad = parent_grad if parent.grad is None else parent.grad + parent_grad


def parameter(*shape, rng, scale=None):
    """A trainable weight, initialised with N(0, scale^2); default scale is He's sqrt(2 / fan_in)."""
    scale = np.sqrt(2.0 / shape[0]) if scale is None else scale
    return Tensor(rng.normal(0.0, scale, shape), requires_grad=True)


def zeros_parameter(*shape):
    return Tensor(np.zeros(shape), requires_grad=True)


def zero_grad(params):
    for p in params:
        p.grad = None


def cross_entropy(logits, targets, mask=None):
    """Mean next-token/class cross-entropy. logits: (..., classes); targets: int (...); mask: 1.0 keeps a position."""
    flat = logits.reshape(-1, logits.shape[-1]).log_softmax(axis=-1)
    targets = np.asarray(targets).reshape(-1)
    picked = flat[np.arange(targets.size), targets]
    if mask is None:
        return -picked.mean()
    weights = np.asarray(mask, dtype=np.float64).reshape(-1)
    return -(picked * weights).sum() / max(weights.sum(), 1.0)


def layer_norm(x, gamma, beta, eps=1e-5):
    mean = x.mean(axis=-1, keepdims=True)
    centred = x - mean
    var = (centred ** 2).mean(axis=-1, keepdims=True)
    return centred / (var + eps) ** 0.5 * gamma + beta


def numerical_grad(f, tensor, eps=1e-6):
    """Central-difference gradient of scalar f() with respect to tensor.data."""
    grad = np.zeros_like(tensor.data)
    it = np.nditer(tensor.data, flags=["multi_index"])
    for _ in it:
        index = it.multi_index
        original = tensor.data[index]
        tensor.data[index] = original + eps
        plus = float(f().data)
        tensor.data[index] = original - eps
        minus = float(f().data)
        tensor.data[index] = original
        grad[index] = (plus - minus) / (2 * eps)
    return grad


def relative_error(a, b):
    return float(np.linalg.norm(a - b) / max(np.linalg.norm(a) + np.linalg.norm(b), 1e-12))

## 📦 Recap: what you built earlier

The second hidden cell re-defines the pieces from Base Model Training under their usual names: `fit_standardiser`, `Linear`, `dropout`, `MLP(sizes, rng, dropout=0.0)` with `parameters()`, `train()` and `eval()`, and `Adam`. Calling `model(Tensor(X))` returns logits of shape `(batch, classes)`; while training with dropout, pass a random generator as well, `model(Tensor(X), rng)`.

In [ ]:
# 📦 Recap: the pieces from Base Model Training, under their usual names.

def fit_standardiser(X):
    """Column means and scales of the TRAINING rows; apply them to every split as (X - mean) / scale."""
    return X.mean(axis=0), np.maximum(X.std(axis=0), 1e-6)


class Linear:
    """A dense layer x @ W + b, with He-initialised W of shape (n_in, n_out) and a zero bias."""

    def __init__(self, n_in, n_out, rng):
        self.W = parameter(n_in, n_out, rng=rng)
        self.b = zeros_parameter(n_out)

    def __call__(self, x):
        return x @ self.W + self.b

    def parameters(self):
        return [self.W, self.b]


def dropout(x, p, training, rng):
    """Inverted dropout: while training, zero each unit with probability p and scale the rest by 1 / (1 - p)."""
    if not training or p == 0.0:
        return x
    keep = (rng.random(x.shape) >= p) / (1.0 - p)
    return x * keep


class MLP:
    """ReLU hidden layers and a linear output layer, e.g. MLP([6, 32, 2], rng). Call it on a Tensor (batch, n_in)."""

    def __init__(self, sizes, rng, dropout=0.0):
        self.layers = [Linear(n_in, n_out, rng) for n_in, n_out in zip(sizes[:-1], sizes[1:])]
        self.p = dropout
        self.training = True

    def __call__(self, x, rng=None):
        for index, layer in enumerate(self.layers):
            x = layer(x)
            if index < len(self.layers) - 1:
                x = dropout(x.relu(), self.p, self.training, rng)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

    def train(self):
        self.training = True

    def eval(self):
        self.training = False


class Adam:
    """Adam over a list of engine parameters: opt.zero_grad(), loss.backward(), opt.step()."""

    def __init__(self, params, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params, self.lr, self.beta1, self.beta2, self.eps = list(params), lr, beta1, beta2, eps
        self.m = [np.zeros_like(p.data) for p in self.params]
        self.v = [np.zeros_like(p.data) for p in self.params]
        self.t = 0

    def zero_grad(self):
        zero_grad(self.params)

    def step(self):
        self.t += 1
        for p, m, v in zip(self.params, self.m, self.v):
            if p.grad is None:
                continue
            m[...] = self.beta1 * m + (1 - self.beta1) * p.grad
            v[...] = self.beta2 * v + (1 - self.beta2) * p.grad ** 2
            m_hat = m / (1 - self.beta1 ** self.t)
            v_hat = v / (1 - self.beta2 ** self.t)
            p.data -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

## 1 · Two related tasks

Both tasks live in the same 20-dimensional input space. Six fixed, perpendicular directions $q_1, \dots, q_6$ act as hidden **detectors**: detector $j$ responds to an input $x$ with $r_j = |x \cdot q_j|$.

- **Source task** (6 classes, plenty of labels): which detector responds most strongly, $y = \arg\max_j r_j$. We have 1,500 labelled training rows and 500 validation rows.
- **Target task** (2 classes, **40 labels**): is the combination $r_1 + r_2 - r_3 + 0.5\,r_4 - 0.5\,r_5$ above its median? We have 40 labelled rows and 500 held-out rows to measure the result.

The tasks share their ingredients, the detector responses, but ask different questions. A network that has learned to compute the responses for the source task should make the target task easy. That is the bet transfer learning makes, and Section 6 tests it.

In [ ]:
DIM = 20
directions, _ = np.linalg.qr(np.random.default_rng(5).normal(size=(DIM, 6)))  # (20, 6), perpendicular unit columns
mix = np.array([1.0, 1.0, -1.0, 0.5, -0.5, 0.0])


def detector_responses(X):
    """|x . q_j| for every row and detector: (n, 20) -> (n, 6)."""
    return np.abs(X @ directions)


def make_source_data(n, seed):
    X = np.random.default_rng(seed).normal(size=(n, DIM))
    return X, detector_responses(X).argmax(axis=1)


TARGET_CUT = np.median(detector_responses(np.random.default_rng(1234).normal(size=(4000, DIM))) @ mix)


def make_target_data(n, seed):
    X = np.random.default_rng(seed).normal(size=(n, DIM))
    return X, (detector_responses(X) @ mix > TARGET_CUT).astype(int)


X_source, y_source = make_source_data(1500, seed=1)
X_source_val, y_source_val = make_source_data(500, seed=2)
X_target, y_target = make_target_data(40, seed=100)
X_target_test, y_target_test = make_target_data(500, seed=3)
print("source classes:", np.bincount(y_source), "  target labels:", np.bincount(y_target), " held-out:", np.bincount(y_target_test))

The inputs are already standard normal, so no standardiser is needed in this notebook. The source classes are roughly balanced, and the target task is a fair coin: a model that guesses scores about 50%.

## 2 · Learning-rate warm-up and cosine decay

The learning rate $\eta$ is the size of each step. At the start of training the weights are random and the gradients can be large and erratic, so a big step can throw the network somewhere it never recovers from. Near the end, a big step keeps bouncing around a minimum instead of settling into it. A common schedule handles both ends: **linear warm-up** over the first $W$ updates, then **cosine decay** to a floor $\eta_{\min}$ by update $T$. Counting updates from $s = 0$:

$$\eta(s) = \begin{cases} \eta_{\text{peak}} \, \dfrac{s + 1}{W} & s < W, \\[2ex] \eta_{\min} + \tfrac{1}{2} (\eta_{\text{peak}} - \eta_{\min}) \big(1 + \cos(\pi p)\big), \quad p = \min\!\Big(\dfrac{s - W}{\max(T - W,\, 1)},\ 1\Big) & s \ge W. \end{cases}$$

The clock $s$ is the **optimiser update**, not the epoch and not the micro-batch. If several micro-batches make up one update (Section 4), the schedule must still advance once per update.

In [ ]:
for p in [0.0, 0.25, 0.5, 0.75, 1.0]:
    print(f"progress {p:4.2f}: cosine factor 0.5 * (1 + cos(pi * p)) = {0.5 * (1 + np.cos(np.pi * p)):.3f}")

### ✍️ Your turn: `warmup_cosine_lr`

Write `warmup_cosine_lr(step, total_steps, peak_lr, warmup_steps, min_lr=0.0)`:

- **inputs:** the update index `step` (counting from 0), the planned number of updates `total_steps`, the peak rate, the number of warm-up updates and the floor;
- **output:** the learning rate for that update, as a Python `float`, following the formula above. With `warmup_steps=0` there is no warm-up; after `total_steps` the rate stays at `min_lr`.

Hints: one `if` for the warm-up branch; `min(..., 1.0)` caps the progress; `np.cos` and `np.pi` do the rest.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def warmup_cosine_lr_reference(step, total_steps, peak_lr, warmup_steps, min_lr=0.0):
    if step < warmup_steps:
        return peak_lr * (step + 1) / warmup_steps
    progress = min((step - warmup_steps) / max(total_steps - warmup_steps, 1), 1.0)
    return float(min_lr + 0.5 * (peak_lr - min_lr) * (1.0 + np.cos(np.pi * progress)))

In [ ]:
def warmup_cosine_lr(step, total_steps, peak_lr, warmup_steps, min_lr=0.0):
    """Learning rate for optimiser update `step` (from 0): linear warm-up, then cosine decay to min_lr."""
    # ✍️ Your code here (about four lines)
    raise NotImplementedError

In [ ]:
warmup_cosine_lr = practice("warmup_cosine_lr", warmup_cosine_lr, warmup_cosine_lr_reference, cases=[
    (0, 100, 0.1, 10),
    (9, 100, 0.1, 10),
    (55, 100, 0.1, 10, 0.001),
    (150, 100, 0.1, 10, 0.001),  # past the end: stays at the floor
    (0, 100, 0.1, 0),            # edge case: no warm-up at all
])
schedule = [warmup_cosine_lr(s, 600, 0.2, 30) for s in range(600)]
plt.figure(figsize=(6, 2.8))
plt.plot(schedule)
plt.xlabel("optimiser update"); plt.ylabel("learning rate"); plt.title("30 warm-up updates, then cosine decay")
plt.show()

**Expected behaviour.** The first update uses $0.1 \cdot 1/10 = 0.01$, update 9 reaches the peak $0.1$, and halfway through the decay (update 55 of 100 with 10 warm-up updates) the rate is the average of peak and floor, $0.0505$. Past the end it stays at the floor. The plot climbs for 30 updates and then follows half a cosine wave down to zero.

## 3 · Gradient clipping by global norm

One bad batch can produce a gradient hundreds of times larger than usual, and a single step along it can wreck weights that took hundreds of updates to learn. **Clipping by global norm** treats all parameter gradients $g_1, \dots, g_k$ as one long vector, measures its length

$$\lVert g \rVert = \sqrt{\textstyle\sum_{j} \sum g_j^2},$$

and, if that exceeds `max_norm`, multiplies **every** gradient by the same factor

$$c = \min\!\Big(1,\ \frac{\text{max\_norm}}{\lVert g \rVert + 10^{-6}}\Big).$$

Using one factor keeps the update's direction and only shortens it. The small constant avoids dividing by zero; frameworks such as PyTorch use the same formula in `torch.nn.utils.clip_grad_norm_`. Clipping limits the *damage* of a bad gradient. It does not fix the cause, which is why the norm should be logged at every update.

**Shapes.** `grads` is a list of NumPy arrays of any shapes, one per parameter. The function returns a **new** list of arrays with the same shapes, and the norm measured *before* clipping.

In [ ]:
demo_grads = [np.array([[3.0, 0.0], [0.0, 0.0]]), np.array([4.0, 0.0])]
norm = np.sqrt(sum(np.sum(g ** 2) for g in demo_grads))
print("global norm:", norm, "  per-parameter norms:", [float(np.linalg.norm(g)) for g in demo_grads])

### ✍️ Your turn: `clip_by_global_norm`

Write `clip_by_global_norm(grads, max_norm)`:

- **inputs:** `grads`, a list of arrays; `max_norm`, a positive number (it may be `np.inf`, meaning "never clip");
- **output:** `(clipped, norm)`: a new list of arrays scaled by $c$, and the global norm before clipping as a Python `float`.

Do not modify the arrays you are given: build new ones with `g * c`. The check calls your function and the reference on the same arrays, and the training loop relies on the original gradients staying intact.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference is in the collapsed cell below.

In [ ]:
def clip_by_global_norm_reference(grads, max_norm):
    norm = float(np.sqrt(sum(np.sum(g ** 2) for g in grads)))
    scale = min(1.0, max_norm / (norm + 1e-6))
    return [g * scale for g in grads], norm

In [ ]:
def clip_by_global_norm(grads, max_norm):
    """Scale all gradients by one factor so that their global norm is at most max_norm.
    grads: list of arrays. Returns (new list of arrays, norm before clipping)."""
    # ✍️ Your code here (three lines)
    raise NotImplementedError

In [ ]:
clip_by_global_norm = practice("clip_by_global_norm", clip_by_global_norm, clip_by_global_norm_reference, cases=[
    (demo_grads, 1.0),
    ([np.array([[1.0, 2.0], [2.0, 0.0]]), np.array([4.0])], 10.0),  # already short enough: unchanged
    ([np.array([0.5, -0.5]), np.ones((2, 3))], np.inf),               # max_norm = inf: never clip
    ([np.zeros(3)], 1.0),                                             # edge case: a zero gradient
])
clipped, norm = clip_by_global_norm(demo_grads, 1.0)
print("clipped:", [g.round(3).tolist() for g in clipped], " norm before:", norm,
      " norm after:", round(float(np.sqrt(sum(np.sum(g ** 2) for g in clipped))), 6))

**Expected behaviour.** The demo gradients have global norm 5, so with `max_norm=1` both are multiplied by $1/5$: the 3 becomes 0.6 and the 4 becomes 0.8, and the new global norm is 1. Each parameter's gradient alone has a smaller norm (3 and 4); only the combined vector exceeds the limit. Gradients that are already short enough come back unchanged.

## 4 · Gradient accumulation

Sometimes the batch you want does not fit in memory. **Gradient accumulation** splits it into micro-batches, runs forward and backward on one micro-batch at a time, adds up the gradients, and updates the weights once at the end. The engine already adds gradients when `backward()` runs twice without `zero_grad`, just as frameworks such as PyTorch do, so the only question is how to scale each micro-batch's loss.

The large-batch loss is the mean over all $n$ rows. A micro-batch $B_k$ with $n_k$ rows has its own mean loss $\bar L_k$, and

$$\bar L = \frac{1}{n} \sum_{i=1}^{n} \ell_i = \sum_k \frac{n_k}{n} \, \bar L_k .$$

So each micro-batch's mean loss must be weighted by its **share of the rows**, $n_k / n$, before `backward()`. Dividing every micro-batch's loss by the *number* of micro-batches is the same thing only when all micro-batches are equally large. With 10 rows and micro-batches of 4 there are three groups of 4, 4 and 2 rows, and the last one needs weight $2/10$, not $1/3$.

In [ ]:
per_example = np.array([0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0])  # pretend per-example losses
groups = [per_example[0:4], per_example[4:8], per_example[8:10]]
print("true mean over 10 rows:      ", per_example.mean())
print("shares n_k / n:              ", sum(len(g) / 10 * g.mean() for g in groups))
print("divide by number of groups:  ", sum(g.mean() / 3 for g in groups), " <- wrong when the groups differ in size")

### ✍️ Your turn: `accumulated_gradients`

Write `accumulated_gradients(params, loss_fn, X, y, micro_batch_size)`:

- **inputs:** `params`, the model's parameter tensors; `loss_fn(X_part, y_part)`, which returns the **mean** loss over the rows it is given, as a scalar `Tensor`; the batch `X` `(n, features)` and `y` `(n,)`; and the micro-batch size;
- **output:** a list of arrays, one per parameter: the gradient of the mean loss over all `n` rows, computed one micro-batch at a time.

Steps: call `zero_grad(params)` first, so that old gradients never leak in; loop `start` over `range(0, n, micro_batch_size)`; take rows `start:stop` with `stop = min(start + micro_batch_size, n)`; call `backward()` on the micro-batch loss multiplied by `(stop - start) / n`; finally return `[p.grad.copy() for p in params]`. Do not update the parameters.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference is in the collapsed cell below.

In [ ]:
def accumulated_gradients_reference(params, loss_fn, X, y, micro_batch_size):
    zero_grad(params)
    n = len(y)
    for start in range(0, n, micro_batch_size):
        stop = min(start + micro_batch_size, n)
        part = loss_fn(X[start:stop], y[start:stop])  # mean loss over this micro-batch
        (part * ((stop - start) / n)).backward()      # weighted by its share of the rows
    return [p.grad.copy() for p in params]

In [ ]:
def accumulated_gradients(params, loss_fn, X, y, micro_batch_size):
    """Gradient of the mean loss over all rows, accumulated micro-batch by micro-batch.
    Returns a list of arrays, one per parameter. Does not change the parameters."""
    # ✍️ Your code here (about six lines)
    raise NotImplementedError

In [ ]:
tiny = MLP([3, 4, 2], np.random.default_rng(0))
tiny_loss = lambda X_part, y_part: cross_entropy(tiny(Tensor(X_part)), y_part)
X_ten = np.random.default_rng(1).normal(size=(10, 3))
y_ten = np.array([0, 1, 1, 0, 1, 0, 0, 1, 1, 0])
accumulated_gradients = practice("accumulated_gradients", accumulated_gradients, accumulated_gradients_reference, cases=[
    (tiny.parameters(), tiny_loss, X_ten, y_ten, 10),
    (tiny.parameters(), tiny_loss, X_ten, y_ten, 4),  # groups of 4, 4 and 2: the short last group
    (tiny.parameters(), tiny_loss, X_ten, y_ten, 1),  # edge case: one row at a time
])

Now the proof. The cell compares the accumulated gradient with the gradient of one large batch, computed in a single `backward()`, and then repeats the comparison with the tempting mistake of dividing each micro-batch loss by the number of micro-batches.

In [ ]:
def wrongly_normalised(params, loss_fn, X, y, micro_batch_size):
    """The common mistake: divide every micro-batch loss by the number of micro-batches."""
    zero_grad(params)
    starts = range(0, len(y), micro_batch_size)
    for start in starts:
        (loss_fn(X[start:start + micro_batch_size], y[start:start + micro_batch_size]) / len(starts)).backward()
    return [p.grad.copy() for p in params]


def flat(grads):
    return np.concatenate([g.ravel() for g in grads])


zero_grad(tiny.parameters())
tiny_loss(X_ten, y_ten).backward()
large_batch = [p.grad.copy() for p in tiny.parameters()]
error_right = relative_error(flat(accumulated_gradients(tiny.parameters(), tiny_loss, X_ten, y_ten, 4)), flat(large_batch))
error_wrong = relative_error(flat(wrongly_normalised(tiny.parameters(), tiny_loss, X_ten, y_ten, 4)), flat(large_batch))
print(f"micro-batches 4, 4, 2 weighted by n_k / n: relative error {error_right:.1e}")
print(f"micro-batches 4, 4, 2 divided by 3:        relative error {error_wrong:.1e}")

**Expected behaviour.** Weighted by their shares, the micro-batches reproduce the large-batch gradient to about $10^{-16}$, which is rounding error. Dividing by the number of micro-batches gives the two-row group as much weight as each four-row group, and the gradient is off by a few per cent. The same holds only while the model behaves identically in both runs: with dropout active, every forward pass draws new random masks, so compare accumulation with a full batch only with dropout off.

## 5 · One training loop with all three controls

`train` below uses your three functions on every update: it accumulates the gradient over micro-batches, measures (and optionally clips) its global norm, looks up the learning rate for this update, and takes one SGD-with-momentum step. It records the loss, the learning rate and the gradient norm of every update, because you cannot diagnose instability you have not logged. The `corrupt_at` option plants one bad record for the experiment in Section 7.

In [ ]:
def sgd_momentum_step(params, grads, velocities, lr, momentum=0.9):
    """v <- momentum * v + g, then p <- p - lr * v, for every parameter (in place)."""
    for p, g, v in zip(params, grads, velocities):
        v *= momentum
        v += g
        p.data -= lr * v


def accuracy(model, X, y):
    model.eval()
    return float(np.mean(model(Tensor(X)).data.argmax(axis=1) == y))


def train(model, X, y, updates=600, batch_size=64, micro_batch_size=None, peak_lr=0.2, warmup_steps=30,
          use_schedule=True, max_norm=np.inf, seed=1, corrupt_at=None):
    """SGD with momentum on random batches. Returns a log of loss, learning rate and gradient norm per update."""
    params = model.parameters()
    velocities = [np.zeros_like(p.data) for p in params]
    g = np.random.default_rng(seed)
    loss_fn = lambda X_part, y_part: cross_entropy(model(Tensor(X_part), g), y_part)
    log = {"loss": [], "lr": [], "norm": []}
    for step in range(updates):
        rows = g.integers(0, len(y), size=batch_size)
        X_batch, y_batch = X[rows], y[rows]
        if step == corrupt_at:  # one bad record: features 100 times too large and the wrong label
            X_batch, y_batch = X_batch.copy(), y_batch.copy()
            X_batch[0] *= 100.0
            y_batch[0] = (y_batch[0] + 3) % 6
        model.train()
        grads = accumulated_gradients(params, loss_fn, X_batch, y_batch, micro_batch_size or batch_size)
        grads, norm = clip_by_global_norm(grads, max_norm)
        lr = warmup_cosine_lr(step, updates, peak_lr, warmup_steps) if use_schedule else peak_lr
        sgd_momentum_step(params, grads, velocities, lr)
        model.eval()
        log["loss"].append(float(loss_fn(X_batch, y_batch).data))
        log["lr"].append(lr)
        log["norm"].append(norm)
    return log

**The schedule's clock.** With micro-batches it is easy to call the scheduler once per micro-batch instead of once per update. The cell shows what that does when four micro-batches make one update: the schedule runs four times too fast and reaches its floor a quarter of the way through training.

In [ ]:
updates, micro_per_update = 600, 4
right_clock = [warmup_cosine_lr(u, updates, 0.2, 30) for u in range(updates)]
wrong_clock = [warmup_cosine_lr(micro_per_update * (u + 1) - 1, updates, 0.2, 30) for u in range(updates)]
for u in [0, 50, 150, 300]:
    print(f"update {u:3d}: per update {right_clock[u]:.4f}   per micro-batch {wrong_clock[u]:.4f}")

## 6 · Pretrain, freeze and transfer

First, pretrain a source network on the 6-class task with everything from Sections 2 to 5: batches of 64 rows built from four micro-batches of 16, a warm-up and cosine schedule, and clipping at a generous `max_norm` of 5 (about five times the typical norm, so that it only acts on outliers).

In [ ]:
start = time.perf_counter()
source_model = MLP([DIM, 64, 64, 6], np.random.default_rng(0))
pretrain_log = train(source_model, X_source, y_source, updates=600, batch_size=64, micro_batch_size=16,
                     peak_lr=0.2, warmup_steps=30, max_norm=5.0)
source_accuracy = accuracy(source_model, X_source_val, y_source_val)
print(f"pretrained in {time.perf_counter() - start:.2f}s: source validation accuracy {source_accuracy:.3f} (chance is about 0.17)")
print(f"median gradient norm {np.median(pretrain_log['norm']):.2f}; updates clipped: {np.sum(np.array(pretrain_log['norm']) > 5.0)} of 600")

**Freezing.** To reuse the network we keep everything except its last layer, the **backbone**, and **freeze** it: its parameters stop requiring gradients, so no tape is recorded through them and no optimiser can change them. A new **head**, a single linear layer from the backbone's 64 features to the target's 2 classes, is the only thing trained. It has $64 \times 2 + 2 = 130$ trainable parameters, against about 5,900 in the whole network.

Freezing has a second, easily missed meaning. It switches off *learning*, not *train-mode behaviour*: a frozen network with dropout still drops units at random until you call `eval()`. The demo shows both facts on a small network with dropout.

In [ ]:
def freeze(model):
    for p in model.parameters():
        p.requires_grad = False


def count_trainable(params):
    return sum(p.data.size for p in params if p.requires_grad)


with_dropout = MLP([DIM, 16, 6], np.random.default_rng(7), dropout=0.5)
print("trainable before freezing:", count_trainable(with_dropout.parameters()))
freeze(with_dropout)
print("trainable after freezing: ", count_trainable(with_dropout.parameters()))
out = with_dropout(Tensor(X_source[:2]), np.random.default_rng(8))
print("output records a tape:", out.requires_grad)
with_dropout.train()
first = with_dropout(Tensor(X_source[:2]), np.random.default_rng(8)).data
second = with_dropout(Tensor(X_source[:2]), np.random.default_rng(9)).data
print("frozen, train mode: two calls agree?", np.allclose(first, second))
with_dropout.eval()
print("frozen, eval mode:  two calls agree?", np.allclose(with_dropout(Tensor(X_source[:2])).data, with_dropout(Tensor(X_source[:2])).data))

### ✍️ Your turn: `extract_features`

Write `extract_features(model, X)`, the frozen backbone's view of the data:

- **inputs:** a (frozen) `MLP` and inputs `X` `(n, 20)`;
- **output:** the activations after the **last hidden layer**, a NumPy array `(n, hidden)`: run every layer in `model.layers` except the last, each followed by `.relu()`, and return the `.data` of the result. Apply no dropout: features for a frozen backbone are computed in evaluation behaviour.

Hints: start from `h = Tensor(X)`; `model.layers[:-1]` are the hidden layers; each `layer(h)` computes `h @ W + b`.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference is in the collapsed cell below.

In [ ]:
def extract_features_reference(model, X):
    h = Tensor(X)
    for layer in model.layers[:-1]:
        h = layer(h).relu()
    return h.data

In [ ]:
def extract_features(model, X):
    """Activations of the last hidden layer, without dropout: (n, n_in) -> (n, hidden) NumPy array."""
    # ✍️ Your code here (about four lines)
    raise NotImplementedError

In [ ]:
freeze(source_model)
extract_features = practice("extract_features", extract_features, extract_features_reference, cases=[
    (source_model, X_source[:5]),
    (MLP([3, 4, 5, 2], np.random.default_rng(3)), np.ones((2, 3))),
    (MLP([3, 2], np.random.default_rng(4)), np.ones((2, 3))),  # edge case: no hidden layer, the features are the inputs
])
features_train = extract_features(source_model, X_target)
features_test = extract_features(source_model, X_target_test)
print("target features:", features_train.shape, features_test.shape)

**Expected behaviour.** The 40 target rows become a `(40, 64)` feature matrix and the 500 held-out rows a `(500, 64)` one. For a network without hidden layers the "features" are just the inputs. Because the backbone is frozen, computing the features records no tape and costs one forward pass.

Now the comparison. Both models get the same 40 labelled rows and the same training budget: 200 updates of 32 rows with a short warm-up. The transferred model trains only the new head on the frozen features; the scratch model trains a fresh network of the same shape as the source network on the raw inputs.

In [ ]:
def transfer_vs_scratch(n_labels, seed):
    X_small, y_small = make_target_data(n_labels, seed=seed)
    head = MLP([64, 2], np.random.default_rng(seed))  # a single linear layer: 130 trainable parameters
    train(head, extract_features(source_model, X_small), y_small, updates=200, batch_size=32, peak_lr=0.05, warmup_steps=10, seed=seed)
    scratch = MLP([DIM, 64, 64, 2], np.random.default_rng(seed))
    train(scratch, X_small, y_small, updates=200, batch_size=32, peak_lr=0.05, warmup_steps=10, seed=seed)
    return accuracy(head, features_test, y_target_test), accuracy(scratch, X_target_test, y_target_test)


transfer_accuracy, scratch_accuracy = transfer_vs_scratch(40, seed=100)
print(f"40 labels: frozen backbone + new head {transfer_accuracy:.3f}, trained from scratch {scratch_accuracy:.3f} (chance about 0.5)")

**What to notice.** With 40 labels, the network trained from scratch barely beats a coin, while the new head on frozen features gets most held-out rows right. The scratch model would have to discover the six detector directions from 40 examples; the backbone already computes something close to them. Transfer helps here because the tasks share their ingredients. On an unrelated task, frozen features can be useless or worse, which is why the frozen-feature baseline comes first and unfreezing comes later, deliberately, with a smaller learning rate for the pretrained layers.

## 7 · 🧪 Experiments

The two helpers below run the controlled comparisons: the same initial weights and the same batches in every run, with exactly one thing changed. The cells after them are yours to edit.

In [ ]:
def compare_schedules(peak_lr, warmup_steps, updates=600):
    """Validation accuracy and mean training loss over the last 50 updates, for a constant rate and for the schedule."""
    out = {}
    for label, use_schedule in [("constant", False), ("warm-up + cosine", True)]:
        model = MLP([DIM, 64, 64, 6], np.random.default_rng(0))
        log = train(model, X_source, y_source, updates=updates, peak_lr=peak_lr, warmup_steps=warmup_steps, use_schedule=use_schedule)
        out[label] = (accuracy(model, X_source_val, y_source_val), float(np.mean(log["loss"][-50:])))
    return out


def corruption_runs(max_norm, bad_update=200, updates=400):
    """A clean run, a run with one corrupted record and no clipping, and the same run clipped at max_norm."""
    out = {}
    for label, corrupt, limit in [("clean", None, np.inf), ("corrupted, no clipping", bad_update, np.inf),
                                  ("corrupted, clipped", bad_update, max_norm)]:
        model = MLP([DIM, 64, 64, 6], np.random.default_rng(0))
        log = train(model, X_source, y_source, updates=updates, peak_lr=0.1, warmup_steps=20, max_norm=limit, corrupt_at=corrupt)
        out[label] = (log, accuracy(model, X_source_val, y_source_val))
    return out

### 🧪 A constant rate versus warm-up and cosine decay

One run keeps `peak_lr` for all 600 updates; the other warms up and decays. Change the values and run the cell again.

In [ ]:
peak_lr = 0.2       # 🧪 try 0.05 and 0.4
warmup_steps = 30   # 🧪 try 0 (decay only)
results = compare_schedules(peak_lr, warmup_steps)
for label, (val_accuracy, late_loss) in results.items():
    print(f"{label:>16}: training loss over the last 50 updates {late_loss:.3f}, validation accuracy {val_accuracy:.3f}")

At `peak_lr = 0.2` the constant rate keeps bouncing: its training loss stays high and its validation accuracy lags well behind the scheduled run. At 0.05 the constant rate does as well as the schedule or slightly better, because a small rate is already gentle. At 0.4 the constant rate fails outright, while the schedule still learns. Removing the warm-up changes little on this small network, where the decay does most of the work; on large models, early instability is where warm-up earns its place.

### 🧪 One corrupted record, with and without clipping

At update 200, one row of the batch arrives with features 100 times too large and the wrong label, the kind of record that a unit mistake plus a labelling error produce. Three runs: a clean one, a corrupted one without clipping, and a corrupted one clipped at `max_norm`.

In [ ]:
max_norm = 5.0  # 🧪 try 1.0 and 50.0
runs = corruption_runs(max_norm)
for label, (log, val_accuracy) in runs.items():
    clipped_updates = int(np.sum(np.array(log["norm"]) > max_norm)) if "clipped" in label else 0
    print(f"{label:>23}: norm at update 200 {log['norm'][200]:6.2f} (median {np.median(log['norm']):.2f}), "
          f"mean loss over the next 20 updates {np.mean(log['loss'][201:221]):.3f}, "
          f"validation accuracy {val_accuracy:.3f}, updates clipped {clipped_updates}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for label, (log, _) in runs.items():
    axes[0].plot(log["norm"], label=label, linewidth=1)
    axes[1].plot(log["loss"], label=label, linewidth=1)
axes[0].set_yscale("log"); axes[0].set_xlabel("update"); axes[0].set_ylabel("gradient norm (log scale)")
axes[1].set_xlabel("update"); axes[1].set_ylabel("batch loss"); axes[1].set_ylim(0, 3); axes[1].legend()
plt.tight_layout(); plt.show()

The logged norm jumps to about 50 times its median at update 200 in both corrupted runs: the log points straight at the bad batch. Without clipping, one step along that gradient sends the loss back up and the network spends the rest of training recovering. With clipping, the same step is shortened to `max_norm` and the loss barely moves. Clipping protected the weights, but the corrupted record is still in the data: the fix is to find it and correct or remove it. With `max_norm = 50` the limit is above the spike and nothing changes; with `max_norm = 1.0` the clipper also acts on about a third of all updates, which quietly changes the optimiser instead of only guarding against outliers.

### 🧪 How many labels does transfer save?

Change `n_labels` and run the cell again. Each setting repeats the comparison with three different label draws and reports the mean and the worst of the three.

In [ ]:
n_labels = 40  # 🧪 try 10, 100 and 400
scores = np.array([transfer_vs_scratch(n_labels, seed=200 + k) for k in range(3)])
print(f"{n_labels} labels: transfer {scores[:, 0].mean():.3f} (worst {scores[:, 0].min():.3f}), "
      f"scratch {scores[:, 1].mean():.3f} (worst {scores[:, 1].min():.3f})")

With 10 labels the frozen backbone already beats a coin, but the result depends more on which 10 rows you drew. With 40 and 100 labels transfer wins by about 25 percentage points. With 400 labels the network trained from scratch catches up part of the way, because it now has enough examples to discover the detectors for itself.

## 8 · Put it together

The cell checks the behaviour of everything you built: the schedule's shape, exact gradient accumulation including the short micro-batch, the two controlled comparisons at their default settings (rerun here if you changed the experiments above), and the benefit of transfer with 40 labels.

In [ ]:
schedule_check = results if (peak_lr, warmup_steps) == (0.2, 30) else compare_schedules(0.2, 30)
clip_check = runs if max_norm == 5.0 else corruption_runs(5.0)
lrs = [warmup_cosine_lr(s, 600, 0.2, 30) for s in range(600)]
assert abs(lrs[0] - 0.2 / 30) < 1e-12 and abs(max(lrs) - 0.2) < 1e-12 and lrs[-1] < 0.001, "warm-up then decay to the floor"
assert error_right < 1e-10, "accumulated micro-batches must reproduce the large-batch gradient"
assert error_wrong > 1e-3, "dividing by the number of micro-batches should be visibly wrong for groups of 4, 4 and 2"
assert source_accuracy > 0.75, f"the pretrained source network should be accurate, got {source_accuracy:.3f}"
(acc_constant, loss_constant), (acc_schedule, loss_schedule) = schedule_check["constant"], schedule_check["warm-up + cosine"]
assert acc_schedule > acc_constant + 0.02 and loss_schedule < 0.5 * loss_constant, "the schedule should beat a constant 0.2"
broken, clipped = clip_check["corrupted, no clipping"][0], clip_check["corrupted, clipped"][0]
assert broken["norm"][200] > 10 * np.median(broken["norm"]), "the bad record should show up as a spike in the logged norm"
assert np.mean(broken["loss"][201:221]) > 2 * np.mean(clipped["loss"][201:221]), "clipping should contain the damage"
assert transfer_accuracy > 0.65 and transfer_accuracy > scratch_accuracy + 0.1, "transfer should beat scratch with 40 labels"
print("✅ Schedule, clipping, accumulation and transfer all behave as expected.")
print(f"   accumulation error {error_right:.1e} (wrong normalisation {error_wrong:.1e})")
print(f"   validation accuracy at peak 0.2: constant rate {acc_constant:.3f}, scheduled {acc_schedule:.3f}")
print(f"   40 target labels: transfer {transfer_accuracy:.3f}, scratch {scratch_accuracy:.3f}")

## Exercises

### Exercise 1: clip each parameter separately, and see what goes wrong

Write `clip_each_parameter(grads, max_norm)`, which clips every array **on its own**: array $g_j$ is multiplied by $\min(1, \text{max\_norm} / (\lVert g_j \rVert + 10^{-6}))$. Return the new list. The cell after the check compares the direction of the clipped update with the original one, for global and for per-parameter clipping.

In [ ]:
def clip_each_parameter_reference(grads, max_norm):
    return [g * min(1.0, max_norm / (float(np.sqrt(np.sum(g ** 2))) + 1e-6)) for g in grads]

In [ ]:
def clip_each_parameter(grads, max_norm):
    """Clip every gradient array separately to norm max_norm. Returns a new list of arrays."""
    # ✍️ Your code here (one or two lines)
    raise NotImplementedError

In [ ]:
clip_each_parameter = practice("clip_each_parameter", clip_each_parameter, clip_each_parameter_reference, cases=[
    (demo_grads, 1.0),
    ([np.array([0.1, 0.2]), np.array([30.0, 40.0])], 5.0),
    ([np.zeros(2)], 1.0),  # edge case: a zero gradient stays zero
])
original = [np.array([3.0, 0.0]), np.array([0.0, 0.4])]
cosine = lambda a, b: float(flat(a) @ flat(b) / (np.linalg.norm(flat(a)) * np.linalg.norm(flat(b))))
print("cosine with the original after global clipping:       ", round(cosine(clip_by_global_norm(original, 0.5)[0], original), 4))
print("cosine with the original after per-parameter clipping:", round(cosine(clip_each_parameter(original, 0.5), original), 4))

### Exercise 2: step decay

Write `step_decay_lr(step, peak_lr, drop_every, factor=0.1)`, which multiplies the rate by `factor` after every `drop_every` updates (so updates 0 to `drop_every - 1` use `peak_lr`). Replace the schedule in the first experiment with it and compare.

In [ ]:
def step_decay_lr_reference(step, peak_lr, drop_every, factor=0.1):
    return float(peak_lr * factor ** (step // drop_every))

In [ ]:
def step_decay_lr(step, peak_lr, drop_every, factor=0.1):
    """peak_lr * factor ** (number of completed drop_every-update blocks)."""
    # ✍️ Your code here (one line)
    raise NotImplementedError

In [ ]:
step_decay_lr = practice("step_decay_lr", step_decay_lr, step_decay_lr_reference, cases=[
    (0, 0.2, 100), (99, 0.2, 100), (100, 0.2, 100), (250, 0.2, 100, 0.5),
])

### Exercise 3 (question): unfreeze deliberately

Unfreeze the backbone (set `requires_grad = True` again) and fine-tune the whole network on the 40 target rows, once with `peak_lr = 0.05` for every parameter and once with a ten times smaller rate for the backbone than for the head (you will need two calls to `sgd_momentum_step` with different rates). Which keeps more of the held-out accuracy, and why can full fine-tuning on 40 rows *lose* accuracy?

### Exercise 4 (question): the linear scaling rule

Doubling the batch size halves the noise in each gradient. A popular heuristic says: double the learning rate too. Test it on the source task with batch sizes 32, 64 and 128 and rates 0.1, 0.2 and 0.4 (same number of *examples* seen in each run). Does the rule hold here at all three sizes? Why is it called a heuristic rather than a law?

## Common errors and debugging

- **Advancing the scheduler per micro-batch.** With $k$ micro-batches per update the schedule runs $k$ times too fast. Step it once per optimiser update and log the rate you actually used.
- **Accumulating without rescaling, or rescaling by the number of micro-batches.** Weight each micro-batch's mean loss by $n_k / n$; only equal micro-batches allow dividing by their number.
- **Forgetting `zero_grad` before the first micro-batch.** The previous update's gradients are then added to this one's.
- **Clipping each parameter separately.** Per-parameter clipping changes the update's direction; global-norm clipping only shortens it.
- **A `max_norm` near the typical norm.** Then clipping acts on almost every update and changes the optimiser. Look at the logged norms and set the limit well above their median.
- **Treating clipping as the fix.** A spike in the norm is a symptom; find the batch, the rows and the cause.
- **Comparing accumulation with a full batch while dropout is on.** The random masks differ between the runs, so the gradients differ too.
- **"Frozen" but still in train mode.** `requires_grad = False` stops updates; dropout and similar layers still behave as in training until you call `eval()`.
- **Mismatched preprocessing.** A pretrained backbone expects its inputs prepared exactly as during pretraining. A different scaling or tokeniser can ruin the features before any adaptation starts.

## Summary

A schedule changes the step size over the optimiser-update clock: warm-up protects the start, cosine decay lets the end settle. Clipping by global norm shortens an update without turning it, and the logged norm shows where trouble came from. Accumulation reproduces the large-batch gradient exactly when each micro-batch is weighted by its share of the rows. Transfer reused a network trained on 1,500 source labels to solve a related task from 40 labels, where training from scratch could not. All of this ran on small networks and generated data: real systems add far larger models, data and hardware, and with them memory limits, which notebook 3 measures.

## Next

**03 · Memory and model selection**: see what float16 does to small gradients and how loss scaling rescues them, trade compute for memory with activation checkpointing, and run a bounded random search with a fixed validation protocol.